# Data Cleaning and Tidying

### Importing Libraries

In [1]:
import pandas as pd
import re

As the initial step in our data cleaning, we standardized the dataset's features by converting all column names to lowercase and identifying any special characters. Upon programmatic inspection, the only special character detected was a period (.) within the num.co column. As demonstrated in the code cell below, this was systematically replaced with an underscore (_).

In [3]:
dirty = pd.read_csv('dataset.csv')
col = dirty.columns.tolist()
specials = set();
for el in col:
    new = re.findall(r'[^a-zA-Z0-9_]', el)
    specials.update(new)

print(specials) # Special char found in columns name, just '.'
df = dirty.rename(columns=lambda c: c.strip().
    lower().
        replace(' ', '_').
            replace('.','_'))


{'.'}


To prevent our algorithms from mistaking arbitrary placeholders (like -999, "unknown", or empty strings) for actual clinical measurements, we must standardize how missing data is represented. This code systematically scans the dataset and replaces all inconsistent sentinel values with Pandas' native missing indicator (pd.NA), ensuring our subsequent imputation tools recognize exactly where the true data gaps are located

In [3]:
df = df.replace({"": pd.NA, "NA": pd.NA, "NaN": pd.NA,"-999":pd.NA, -999 : pd.NA,"unknown": pd.NA})

Due to the absence of a unique patient identifier (e.g., a Patient ID) in this dataset, we cannot definitively track multiple hospital admissions for the same individual. Consequently, we can only identify and remove duplicate records if two rows are perfectly identical across every single clinical feature

In [4]:
df[df.duplicated(subset=df.columns, keep=False)]
# Not exact records match

,age,sex,num_co,diabetes,dementia,meanbp,hrt,resp,temp,crea,...,scoma,dzgroup,aps,wblc,pafi,income,race,adls,hospdead,dnr


This code block executes a multi-step data cleaning pipeline to ensure structural consistency and biological validity across the dataset. First, all categorical and textual columns are standardized by stripping extraneous whitespace and converting them to lowercase. Next, to reduce unnecessary dimensionality, any completely empty columns are dropped. For numerical features, the script forces strict data type coercion (pd.to_numeric), converting any hidden string artifacts into missing values (NaN) rather than halting execution. Finally, we apply a domain-specific logical check: clinical parameters that cannot physically be negative (such as age, heart rate, or creatinine levels) are evaluated. Any erroneous negative entries within these specific columns are masked and replaced with pd.NA, preparing them for proper handling during the imputation phase.

In [5]:
text_col = df.select_dtypes(include=['object', 'string'])
for col in text_col:
    df[col]=df[col].astype(str).str.strip().str.lower()

df = df.dropna(axis=1,how="all") # If the column is completely empty just drop it

num_col= df.select_dtypes(include=['number']).columns.tolist()
for col in num_col:
    df[col]=pd.to_numeric(df[col],errors="coerce").convert_dtypes()

not_neg_col = ["age","num_co","diabetes","dementia","meanbp",
               "hrt","resp","temp","pafi","crea","sod",
               "scoma","aps","hday","adls","adlsc"]

for col in not_neg_col:
    df[col] = df[col].mask(df[col] < 0, pd.NA )


To ensure the integrity of our dataset, this code block applies strict, domain-specific logical constraints to both categorical and numerical features. For categorical variables (such as sex, race, income, ca, and dnr), we defined exhaustive lists of valid, expected strings; any entry falling outside these accepted vocabularies is systematically identified using boolean masking (~df.isin()) and converted to pd.NA. Similarly, physiological boundaries are enforced on continuous variables (e.g., white blood cell count wblc must be strictly positive, and age must be realistically capped below 120). Finally, boolean clinical indicators (like diabetes and dementia) are strictly constrained to 0 or 1 and explicitly cast to Pandas' nullable integer type (Int64), preventing the injection of floating-point artifacts during the imputation phase.

Why is not the scoma value between 3-15? 

Although the README.md documentation specifies that the scoma value should range from 3 to 15, the observed values in our dataset range from 0 to 100. Over 50% of these values are exactly 0; however, given that this dataset comprises patients with severe health conditions, this heavy concentration at 0 is expected. Because the remaining values are well-distributed between 9 and 100, we assume this variable remains a valid metric for assessing a patient's level of consciousness.

In [6]:
# SEX
valid_sex=["male","female"]
df.loc[~df['sex'].isin(valid_sex), 'sex'] = pd.NA

# WBLC
df.loc[~(df['wblc'] > 0), 'wblc'] = pd.NA

# AGE 
df.loc[~(df['age'] < 120), 'age'] = pd.NA

# RACE
valid_race=["white","black","asian","hispanic","other"]
df.loc[~df['race'].isin(valid_race), 'race'] = pd.NA

# INCOME
valid_income = ["<$11k","$11-$25k","$25-$50k",">$50k","under $11k"]
df.loc[~df['income'].isin(valid_income), 'income'] = pd.NA

# CA
valid_ca = ["no","yes","metastatic"]
df.loc[~df['ca'].isin(valid_ca), 'ca'] = pd.NA

# DNR
valid_dnr = ["no dnr","dnr before sadm","dnr after sadm"]
df.loc[~df['dnr'].isin(valid_dnr), 'dnr'] = pd.NA

# DIABETES, DEMENTIA  (BIN VALUES 0,1)
bin_col = ["dementia","diabetes"]
valid_bin = [0,1]
for col in bin_col:
    df.loc[~df[col].isin(valid_bin), col] = pd.NA
    df[col] = df[col].astype("Int64")


As illustrated in the output below, the highest proportion of missing data is concentrated within the pafi and income features, both of which exhibit a missingness rate of approximately 50%

In [7]:
df.info()
df.to_csv('dataset_clean_tidy.csv', index=False)

<class 'pandas.DataFrame'>
RangeIndex: 9105 entries, 0 to 9104
Data columns (total 24 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       8650 non-null   Float64
 1   sex       9105 non-null   str    
 2   num_co    9105 non-null   Int64  
 3   diabetes  9105 non-null   Int64  
 4   dementia  9105 non-null   Int64  
 5   meanbp    9104 non-null   Float64
 6   hrt       9104 non-null   Float64
 7   resp      9104 non-null   Int64  
 8   temp      9104 non-null   Float64
 9   crea      9038 non-null   Float64
 10  sod       9104 non-null   Int64  
 11  adlsc     9105 non-null   Float64
 12  hday      9105 non-null   Int64  
 13  ca        9105 non-null   str    
 14  scoma     9104 non-null   Int64  
 15  dzgroup   9105 non-null   str    
 16  aps       9104 non-null   Int64  
 17  wblc      8883 non-null   Float64
 18  pafi      5919 non-null   Float64
 19  income    5008 non-null   str    
 20  race      9063 non-null   str    
 21  ad